<a href="https://colab.research.google.com/github/albijanashala/ML1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
%pip -q install duckdb

import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected")

Connected


In [2]:
DECISION_DATE = "2026-02-28"

features = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks) AS clicks_feb,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_visible_feb,
               SUM(CASE WHEN report_date <= DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h1,
               SUM(CASE WHEN report_date >  DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h2,
               MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS has_ga4
        FROM {FEB}
        GROUP BY 1, 2
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_mar
        FROM {MAR}
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_feb,
        f.clicks_feb,
        f.days_visible_feb,
        ROUND(f.clicks_feb::FLOAT / NULLIF(f.imp_feb, 0), 5) AS ctr_feb,
        ROUND(f.imp_feb::FLOAT / NULLIF(f.days_visible_feb, 0), 2) AS imp_per_active_day,
        ROUND(f.imp_h2::FLOAT / NULLIF(f.imp_h1, 0), 3) AS trend_within_feb,
        f.has_ga4,
        COALESCE(d.word_count, 0) AS word_count,
        CASE WHEN d.word_count IS NULL THEN 1 ELSE 0 END AS word_count_missing,
        COALESCE(d.search_volume, 0) AS search_volume,
        COALESCE(d.competition, 0) AS competition,
        COALESCE(d.backlinks, 0) AS backlinks,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
        COALESCE(d.content_type, 'unknown') AS content_type,
        COALESCE(d.competition_level, 'unknown') AS competition_level,
        COALESCE(d.main_intent, 'unknown') AS main_intent,
        COALESCE(m.imp_mar, 0) AS imp_mar
    FROM feb f
    LEFT JOIN mar m USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE f.imp_feb >= 50
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

print(f"Rows: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 93,654


,client_hash_id,content_hash_id,imp_feb,clicks_feb,days_visible_feb,ctr_feb,imp_per_active_day,trend_within_feb,has_ga4,word_count,word_count_missing,search_volume,competition,backlinks,content_age_days,content_type,competition_level,main_intent,imp_mar
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,246.0,1.0,28,0.00407,8.79,0.662,0,3168,0,0,0.00,0,144,keyword article,LOW,informational,331.0
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,137.0,0.0,27,0.00000,5.07,1.076,0,4135,0,20,0.03,9,144,keyword article,LOW,commercial,33.0
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,121.0,0.0,28,0.00000,4.32,0.833,0,3211,0,0,0.00,0,144,keyword article,LOW,informational,145.0
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,174.0,0.0,12,0.00000,14.50,NaN,0,3465,0,0,0.00,0,144,keyword article,LOW,informational,461.0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,164.0,0.0,28,0.00000,5.86,0.451,0,3149,0,0,0.00,0,144,keyword article,LOW,informational,232.0


In [3]:
df = features.copy()

FEB_DAYS, MAR_DAYS = 28, 31
imp_rate_feb = df["imp_feb"] / FEB_DAYS
imp_rate_mar = df["imp_mar"] / MAR_DAYS
df["is_declining"] = (imp_rate_mar < 0.8 * imp_rate_feb).astype(int)

df["no_h1_impressions"] = df["trend_within_feb"].isna().astype(int)
df["trend_within_feb"] = df["trend_within_feb"].fillna(1.0)
df["content_age_days"] = df["content_age_days"].fillna(-1)

print(f"Rows: {len(df):,}")
print(f"Positive rate: {df['is_declining'].mean():.1%} ({df['is_declining'].sum():,} declining)")

Rows: 93,654
Positive rate: 27.6% (25,887 declining)


In [4]:
encoded_prefixes = ("ctype_", "intent_")
df = df.drop(columns=[c for c in df.columns if c.startswith(encoded_prefixes)], errors="ignore")
df = df.drop(columns=["competition_level_ord"], errors="ignore")

competition_order = {"unknown": 0, "LOW": 1, "MEDIUM": 2, "HIGH": 3}
df["competition_level_ord"] = df["competition_level"].map(competition_order).fillna(0).astype(int)

one_hot = pd.get_dummies(df[["content_type", "main_intent"]],
                         prefix=["ctype", "intent"], dtype=int)
df = pd.concat([df, one_hot], axis=1)

drop_cols = ["client_hash_id", "content_hash_id", "imp_mar", "is_declining",
             "content_type", "competition_level", "main_intent"]
model_features = [c for c in df.columns if c not in drop_cols]

print(f"Model-ready features: {len(model_features)}")

Model-ready features: 23


In [5]:
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

OUT_DIR = Path("work/outputs")
FIG_DIR = Path("work/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

X = df[model_features]
y = df["is_declining"]
groups = df["client_hash_id"]

# Out-of-fold predictions: every page scored by a model that never saw its client
df["model_score"] = np.nan
for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                                   random_state=42, n_jobs=-1)
    model.fit(X.iloc[tr], y.iloc[tr])
    df.iloc[te, df.columns.get_loc("model_score")] = model.predict_proba(X.iloc[te])[:, 1]

# Hand rule from Week 5, kept as a sanity column
momentum_risk = (1 - df["trend_within_feb"]).clip(lower=0, upper=1)
visibility_risk = 1 - (df["days_visible_feb"] / 28)
age_risk = (df["content_age_days"].clip(lower=0, upper=365) / 365)
df["rule_score"] = (0.60 * momentum_risk + 0.25 * visibility_risk + 0.15 * age_risk).round(4)


def reason_code(row):
    if row["trend_within_feb"] < 0.8:
        return "MOMENTUM_LOSS"
    if row["days_visible_feb"] < 20:
        return "INTERMITTENT_VISIBILITY"
    if row["content_age_days"] > 365:
        return "AGEING_CONTENT"
    if row["ctr_feb"] < 0.001:
        return "WEAK_CLICK_CAPTURE"
    return "MODEL_SIGNAL_ONLY"


df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = df["reason_code"].map({
    "MOMENTUM_LOSS": "investigate_decline",
    "INTERMITTENT_VISIBILITY": "check_indexing",
    "AGEING_CONTENT": "content_refresh",
    "WEAK_CLICK_CAPTURE": "snippet_review",
    "MODEL_SIGNAL_ONLY": "manual_triage",
})

# Disagreement flag: model and rule tell different stories about the same page
df["rule_pct"] = df["rule_score"].rank(pct=True)
df["model_pct"] = df["model_score"].rank(pct=True)
df["disagreement"] = (df["model_pct"] - df["rule_pct"]).abs().round(3)
df["needs_human_check"] = (df["disagreement"] > 0.4).astype(int)

# Cost/value: decline risk weighted by what is actually at stake.
# A page with 80,000 impressions falling matters more than one with 55 falling,
# even when the model is more certain about the small page. log1p keeps volume a
# factor without letting the largest pages take over the whole queue.
df["impressions_at_risk"] = (df["model_score"] * df["imp_feb"]).round(0)
df["priority_score"] = (df["model_score"] * np.log1p(df["imp_feb"])).round(4)

queue = df.sort_values("priority_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

print(f"Pages scored: {len(queue):,}")
print("\nAction distribution:")
print(queue["action"].value_counts().to_string())
print(f"\nFlagged for human check: {queue['needs_human_check'].sum():,} "
      f"({queue['needs_human_check'].mean():.1%})")
print(f"\nImpressions at risk, top 50: {queue.head(50)['impressions_at_risk'].sum():,.0f}")
print("\nTop 20 by priority score:")
print(queue[["rank", "priority_score", "model_score", "action", "reason_code",
             "imp_feb", "clicks_feb", "trend_within_feb", "needs_human_check"]]
      .head(20).to_string(index=False))

Pages scored: 93,654

Action distribution:
action
manual_triage          29332
snippet_review         23348
investigate_decline    19946
check_indexing         15485
content_refresh         5543

Flagged for human check: 22,004 (23.5%)

Impressions at risk, top 50: 699,020

Top 20 by priority score:
 rank  priority_score  model_score              action   reason_code  imp_feb  clicks_feb  trend_within_feb  needs_human_check
    1          7.3050     0.923228 investigate_decline MOMENTUM_LOSS   2730.0         3.0             0.338                  0
    2          7.2968     0.724605 investigate_decline MOMENTUM_LOSS  23624.0         9.0             0.210                  0
    3          7.2233     0.658685 investigate_decline MOMENTUM_LOSS  57886.0        82.0             0.154                  0
    4          7.1713     0.648566 investigate_decline MOMENTUM_LOSS  63398.0       148.0             0.180                  0
    5          6.9953     0.663731 investigate_decline MOMENTUM_

In [6]:
print(f"Clients in the eligible frame : {df['client_hash_id'].nunique()}")
print(f"Clients in raw February data  : "
      f"{con.sql(f'SELECT COUNT(DISTINCT client_hash_id) FROM {FEB}').fetchone()[0]}")
print("\nPages per client, eligible frame:")
print(df['client_hash_id'].value_counts().describe().round(1).to_string())

Clients in the eligible frame : 38
Clients in raw February data  : 54

Pages per client, eligible frame:
count       38.0
mean      2464.6
std       4682.7
min          2.0
25%         27.8
50%        532.0
75%       1988.0
max      20358.0


**The queue: what to review first, and why.**

93,654 pages, each scored out-of-fold — every page is ranked by a model that never
saw its own client during training. Ranking is by `priority_score`, which is the
model's decline probability weighted by `log1p(impressions)`.

**Why weight by volume.** Ranking on probability alone put pages with 55 to 600
February impressions and zero clicks at the top. The model was right that they would
decline, but a reviewer gains nothing by fixing a page almost nobody visits. Weighting
by log impressions keeps risk as the driver while letting the size of the loss break
ties. The log matters: multiplying by raw impressions would turn the queue into a
volume ranking, which is the mistake my Week-4 baseline made.

The effect is visible at rank 20 — the highest model score in the top 20 (0.982) but
only 607 impressions. It was rank 1 before weighting.

**Reason codes and their actions.**

| Reason code | Action | Trigger |
|---|---|---|
| `MOMENTUM_LOSS` | investigate_decline | Second half of February below 80% of first half |
| `INTERMITTENT_VISIBILITY` | check_indexing | Fewer than 20 of 28 days with impressions |
| `AGEING_CONTENT` | content_refresh | Over 365 days since creation |
| `WEAK_CLICK_CAPTURE` | snippet_review | CTR below 0.1% |
| `MODEL_SIGNAL_ONLY` | manual_triage | Model flags risk, no single signal dominates |

Across the full queue: manual_triage 29,332, snippet_review 23,348,
investigate_decline 19,946, check_indexing 15,485, content_refresh 5,543.

**A limitation of the reason codes, visible in the top 20.** All twenty carry
`MOMENTUM_LOSS`. The codes are evaluated in order and momentum is checked first, so
any page already losing traction gets that label regardless of what else is true. A
page can be both losing momentum and ageing; the queue only shows the first. The codes
are a triage hint, not a diagnosis, and a reviewer should read the underlying columns
rather than the label alone.

**Disagreement flag.** 22,004 pages (23.5%) sit more than 40 percentile points apart
between the model score and the Week-5 hand rule. None of them are in the top 20 —
where the pages are large and clearly falling, model and rule agree. Disagreement
concentrates further down the list, which is where human judgment is most useful.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this, and for what.**

A content reviewer with limited weekly capacity. They open the queue, work down from
rank 1, and for each page decide whether to act and what to do. The queue answers one
question: *of 93,654 pages, which should a person look at first this week?*

It is decision-support, not a decision. Every row is a suggestion with a reason
attached, and the reason is there so the reviewer can disagree with it.

**Where it stops being valid.**

*It is a ranking tool, not a coverage tool.* Measured on held-out clients in Week 5,
the model produced 300 true positives against 4,651 false negatives — a recall of
roughly 6%. The top of the list is reliable; the list as a whole does not find most
declining pages. Anyone treating this as an inventory of at-risk content would be
wrong about 94% of them.

*It cannot see pages that turn.* The false negatives shared an observed profile:
median `trend_within_feb` of 1.182, meaning pages that were growing through February
and then declined in March. Every feature the model holds says those pages were
healthy. A February-only feature window can only detect decline that had already
started.

*Sixteen clients are absent entirely.* Measured: February contains 54 clients, but the
eligibility gate (`imp_feb >= 50`) leaves 38 in the queue. Smaller or newer clients
drop out completely, so the queue is silent about them rather than saying their pages
are healthy.

*Coverage is very uneven among the 38 that remain.* Median 532 pages per client,
minimum 2, maximum 20,358 — one client accounts for roughly 22% of the entire queue. A
reviewer working the global ranking will spend most of their time on a handful of large
clients. Anyone who needs per-client coverage should rank within client rather than
across the portfolio.

*It is built on one month predicting the next.* Features come from February 2026, the
label from March 2026. Seasonal effects, algorithm updates, and one-off events inside
that window are invisible to it, and nothing in the design accounts for them.

*The proxy label is not an editorial outcome.* `is_declining` is defined as March's
daily impression rate falling below 80% of February's. It is a stand-in for "this page
needs attention", not a measurement of whether anyone should have acted.

*Position is absent by necessity.* The warehouse's position columns produce values
below 1 — for example 3,519 impressions against a `gsc_sum_position` of 524, giving
0.149. Position 1 is the best rank Google awards, so the column could not be trusted
and was excluded. Any ranking-based diagnosis is therefore outside what this queue can
support.

*GA4 coverage is thin.* Only 4.2% of March rows carry `ga4_data_available IS TRUE`, so
engagement, scroll and session signals are not in the feature set. The queue reads
search performance only.

*Model and rule are close.* Across 5 client-grouped folds the Random Forest observed a
higher mean Precision@50 than the Week-5 hand rule (0.836 against 0.680), but the
fold-to-fold spreads overlap. This is a directional preference for the model, not a
demonstrated win.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a person must check before acting.**

*Read the columns, not just the reason code.* The codes are evaluated in order and
momentum is checked first, so all twenty pages at the top of the queue carry
`MOMENTUM_LOSS` even where age or click capture is also poor. The label is a triage
hint; the underlying numbers are the evidence.

*Confirm the decline is real before calling it content decay.* A steep drop inside one
month is as consistent with deindexing, a redirect, a URL change, or cannibalisation
by another page as it is with staleness. Pages 5 and 6 in the top 20 lost more than
75% of their February impressions — that pattern deserves a technical check before an
editorial one.

*Check the 22,004 flagged pages differently.* Where the model and the Week-5 hand rule
disagree by more than 40 percentile points, the two methods are reading the page
differently. Those rows need a judgment call rather than a queue position.

*Treat new pages separately.* A page created inside the feature window has not settled
into its rankings, so low early performance is expected rather than diagnostic.

*Check whether word count is missing before diagnosing thin content.* `word_count = 0`
is my fill for absent metadata, not a measured zero.

**The no-go list — what should never be automated.**

*No automated content changes.* Nothing in this queue should trigger a rewrite, a
metadata change, or a republish without a person reading the page first.

*No automated deletion, merging, or deindexing.* These are irreversible; the queue's
recall of roughly 6% is nowhere near good enough to support them.

*No causal claims about refresh.* This work cannot show that refreshing a page
recovers its traffic. Establishing that needs an experiment with a control group,
which this observational design does not provide.

*No claims about Google's algorithm.* The queue observes associations in one
portfolio over two months.

*No use on clients with no history in the warehouse.* Validation was grouped by client
precisely because client context matters; a client the model has never seen is outside
what was tested.

*No client-facing reporting from raw queue rows.* The scores are internal triage
signals, not a performance grade to show anyone.

*No ranking-based diagnosis.* Position is excluded on data-quality grounds, so the
queue cannot say a page slipped in the results.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**What would tell you the recommendations went stale.**

*The base rate moves.* 27.6% of pages met the decline definition in this window. If a
rerun lands well outside roughly 22–33%, either the portfolio changed or the label
threshold no longer describes the same thing. Investigate before shipping a queue.

*The momentum distribution shifts.* `trend_within_feb` is the model's strongest single
feature — permutation importance of 0.0525, roughly twice the next feature. A
meaningful change in its distribution means the model's main signal is measuring
something different.

*Model and rule start disagreeing more.* 23.5% of pages currently sit more than 40
percentile points apart. A rise suggests the learned pattern is drifting away from the
hand-written one; a sharp fall suggests the model is collapsing toward the rule and
adding little.

*Client coverage changes.* 38 of February's 54 clients pass the eligibility gate. If
that ratio moves, the queue is describing a different portfolio than the one it was
built on — whether because clients joined, left, or crossed the impression threshold.
The warehouse spans 104 clients with `gsc_data_start` dates from 2025-01-27 to
2026-06-02, so new arrivals are expected rather than unusual.

*GA4 coverage changes.* If availability rises well above 4.2%, engagement features
become viable and the feature set should be revisited rather than left as-is.

*Reviewer feedback contradicts the queue.* If reviewers consistently find top-ranked
pages not worth acting on, that is a stronger signal than any metric here.

**Retrain cadence.** Monthly, as each new month closes — the feature and label windows
are monthly, so the natural cadence matches the data. Rebuild rather than update: the
peer medians, the fills, and the model all derive from a single month's population.

**What to monitor rather than retrain on.** The base rate, the action distribution, the
disagreement share, the number of clients passing the eligibility gate, and the number
of pages in the frame. These are cheap to compute and would catch a broken pipeline
before a stale model.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
import json
import matplotlib.pyplot as plt

# The ranked queue: regenerated on every run, blocked from git by design
export_cols = ["rank", "priority_score", "model_score", "rule_score", "action",
               "reason_code", "needs_human_check", "disagreement",
               "client_hash_id", "content_hash_id", "imp_feb", "clicks_feb",
               "ctr_feb", "days_visible_feb", "trend_within_feb",
               "content_age_days", "word_count", "impressions_at_risk"]
queue[export_cols].to_csv(OUT_DIR / "action_playbook_queue.csv", index=False)

# Figure 1: action distribution
fig, ax = plt.subplots(figsize=(7, 4))
counts = queue["action"].value_counts()
ax.barh(counts.index[::-1], counts.values[::-1], color="#4C72B0")
ax.set_xlabel("Pages")
ax.set_title("Action distribution across the ranked queue")
for i, v in enumerate(counts.values[::-1]):
    ax.text(v, i, f" {v:,}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "action_distribution.png", dpi=150)
plt.close(fig)

# Figure 2: why volume weighting changed the queue
fig, ax = plt.subplots(figsize=(7, 4.5))
sample = queue.sample(min(5000, len(queue)), random_state=42)
ax.scatter(sample["imp_feb"], sample["model_score"], s=6, alpha=0.25,
           color="#999999", label="All pages (sample)")
ax.scatter(queue.head(50)["imp_feb"], queue.head(50)["model_score"], s=28,
           color="#C44E52", label="Top 50 by priority score")
ax.set_xscale("log")
ax.set_xlabel("February impressions (log scale)")
ax.set_ylabel("Model decline probability")
ax.set_title("Priority balances risk against what is at stake")
ax.legend(loc="lower left", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "priority_vs_volume.png", dpi=150)
plt.close(fig)

# Metrics: the receipts the paper's numbers trace back to
metrics = {
    "run_date": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "feature_window": "2026-02",
    "label_window": "2026-03",
    "pages_scored": int(len(queue)),
    "clients": int(queue["client_hash_id"].nunique()),
    "base_rate": round(float(df["is_declining"].mean()), 4),
    "scoring": {
        "model": "RandomForestClassifier(n_estimators=200, min_samples_leaf=20)",
        "predictions": "out-of-fold, GroupKFold(5) on client_hash_id",
        "priority_score": "model_score * log1p(imp_feb)",
    },
    "action_counts": queue["action"].value_counts().to_dict(),
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "human_check": {
        "threshold_percentile_gap": 0.4,
        "flagged": int(queue["needs_human_check"].sum()),
        "flagged_share": round(float(queue["needs_human_check"].mean()), 4),
    },
    "impressions_at_risk_top50": int(queue.head(50)["impressions_at_risk"].sum()),
    "known_limits": {
        "recall_held_out_clients": 0.06,
        "false_negative_median_momentum": 1.182,
        "ga4_row_coverage": 0.042,
        "position_columns": "excluded, values below 1 observed",
    },
}

with open(OUT_DIR / "w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Queue   : {OUT_DIR / 'action_playbook_queue.csv'} ({len(queue):,} rows)")
print(f"Figure 1: {FIG_DIR / 'action_distribution.png'}")
print(f"Figure 2: {FIG_DIR / 'priority_vs_volume.png'}")
print(f"Metrics : {OUT_DIR / 'w07_playbook_metrics.json'}")
print()
print(json.dumps(metrics, indent=2))

Queue   : work/outputs/action_playbook_queue.csv (93,654 rows)
Figure 1: work/figures/action_distribution.png
Figure 2: work/figures/priority_vs_volume.png
Metrics : work/outputs/w07_playbook_metrics.json

{
  "run_date": "2026-09-20",
  "feature_window": "2026-02",
  "label_window": "2026-03",
  "pages_scored": 93654,
  "clients": 38,
  "base_rate": 0.2764,
  "scoring": {
    "model": "RandomForestClassifier(n_estimators=200, min_samples_leaf=20)",
    "predictions": "out-of-fold, GroupKFold(5) on client_hash_id",
    "priority_score": "model_score * log1p(imp_feb)"
  },
  "action_counts": {
    "manual_triage": 29332,
    "snippet_review": 23348,
    "investigate_decline": 19946,
    "check_indexing": 15485,
    "content_refresh": 5543
  },
  "reason_code_counts": {
    "MODEL_SIGNAL_ONLY": 29332,
    "WEAK_CLICK_CAPTURE": 23348,
    "MOMENTUM_LOSS": 19946,
    "INTERMITTENT_VISIBILITY": 15485,
    "AGEING_CONTENT": 5543
  },
  "human_check": {
    "threshold_percentile_gap": 0.4,
  

**What this exports, and where each file goes.**

`work/outputs/action_playbook_queue.csv` — the full ranked queue, 93,654 rows. Stays
out of git by design: the CI leak-guard blocks data files, and this notebook
regenerates it on every run.

`work/figures/action_distribution.png` and `work/figures/priority_vs_volume.png` —
committed, because the paper reuses them. The second figure shows why volume weighting
mattered: the top 50 sit in the upper-right region where decline risk and impression
volume are both high, rather than clustering in the high-probability, low-volume corner
where ranking on probability alone had placed them.

`work/outputs/w07_playbook_metrics.json` — committed. Every number quoted in the paper
traces back to this file, including the known limits, so a reader can check a claim
against the run that produced it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.